## PyTorch Baselines on Test Set (ImageNet Val)

Same sweep as `01_1_pytorch_val` but evaluated on the ImageNet validation split (test set).

Results saved under `resultsv2/test_runs/`.

In [ ]:

SPLIT = "val"              # ImageNet val split (= "test" for this project)
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/test/ptq"

In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [11]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from src.config import ExperimentConfig, with_overrides, set_seed
from src.data import get_dataloader
from src.model import get_model
from src.eval import evaluate
from utils.precision import apply_precision
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [12]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
INPUT_BITS = [8, 4, 2, 1]

checkpoints = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            checkpoints[(bits, seed_dir.name)] = str(seed_dir / "best.pth")

print("Checkpoints found:")
for (bits, seed), path in checkpoints.items():
    print(f"  {bits}bit / {seed}: {path}")

Checkpoints found:
  8bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_1/best.pth
  8bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_2/best.pth
  8bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth
  4bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_1/best.pth
  4bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_2/best.pth
  4bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_42/best.pth
  2bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_1/best.pth
  2bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_2/best.pth
  2bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_42/best.pth
  1bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_1bit/seed_1/best

In [13]:
def run_test_experiment(cfg, checkpoint_path, split=SPLIT):
    cfg = cfg.normalized()
    cfg.validate()
    set_seed(cfg)

    criterion = nn.CrossEntropyLoss()
    loader = get_dataloader(cfg, split=split)

    model = apply_precision(get_model(cfg, checkpoint_path=checkpoint_path), cfg)
    t0 = time.perf_counter()
    tracker = evaluate(model, loader, cfg, criterion=criterion)
    elapsed = round(time.perf_counter() - t0, 3)

    payload = {
        "status": "ok",
        "run_id": cfg.run_id(),
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "results": tracker.summary(),
        "total_eval_time_sec": elapsed,
        "split": split,
        "error": None,
    }

    out_path = Path(cfg.output_root) / cfg.run_id() / "result.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    print(f"  Saved: {out_path}")

    return payload

## FP32 Full Sweep (b=8, 4, 2, 1)

In [ ]:
fp32_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    cfg = with_overrides(
        ExperimentConfig(
            backend="pytorch", device="cuda", batch_size=1,
            seed=42, num_eval_batches=None,
            output_root=f"{OUTPUT_ROOT}/{seed}",
        ),
        model_precision="fp32", input_quant_bits=bits,
    )

    result_path = cfg.result_json_path()
    if result_path.exists():
        with open(result_path) as f:
            payload = json.load(f)
        print(f"  SKIP fp32 {bits}bit / {seed} (loaded from disk)")
    else:
        print(f"\n{'='*60}")
        print(f"  {bits}bit / {seed}  |  precision: fp32")
        print(f"{'='*60}")
        payload = run_test_experiment(cfg, checkpoint_path=ckpt_path)

    fp32_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")

## FP16 Full Sweep (b=8, 4, 2, 1)

In [ ]:
fp16_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    cfg = with_overrides(
        ExperimentConfig(
            backend="pytorch", device="cuda", batch_size=1,
            seed=42, num_eval_batches=None,
            output_root=f"{OUTPUT_ROOT}/{seed}",
        ),
        model_precision="fp16", input_quant_bits=bits,
    )

    result_path = cfg.result_json_path()
    if result_path.exists():
        with open(result_path) as f:
            payload = json.load(f)
        print(f"  SKIP fp16 {bits}bit / {seed} (loaded from disk)")
    else:
        print(f"\n{'='*60}")
        print(f"  {bits}bit / {seed}  |  precision: fp16")
        print(f"{'='*60}")
        payload = run_test_experiment(cfg, checkpoint_path=ckpt_path)

    fp16_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")

## Results Summary

In [ ]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"{OUTPUT_ROOT}/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)

display_cols = [
    "seed", "input_bits_trained", "run_id",
    "cfg.model_precision", "cfg.input_quant_bits",
    "res.top1_acc", "res.top5_acc",
    "res.infer_ms_avg", "res.throughput_infer_sps", "res.total_samples",
]

df[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision", "cfg.input_quant_bits"],
    ascending=[False, True, True, False],
)

In [17]:
avg_df = df[df["cfg.backend"] == "pytorch"].groupby(
    ["cfg.backend", "cfg.model_precision", "input_bits_trained"]
).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision", "input_bits_trained"],
    ascending=[True, True],
).reset_index(drop=True)
avg_df

,cfg.backend,cfg.model_precision,input_bits_trained,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,pytorch,fp16,1,69.551,0.475,89.477,0.126,3.065,0.058,326.331
1,pytorch,fp16,2,76.888,0.425,93.139,0.053,3.075,0.046,325.244
2,pytorch,fp16,4,78.330,0.369,93.642,0.101,3.087,0.029,323.959
3,pytorch,fp16,8,77.975,0.101,93.581,0.237,3.048,0.050,328.125
4,pytorch,fp32,1,69.551,0.466,89.470,0.129,2.980,0.013,335.550
5,pytorch,fp32,2,76.861,0.413,93.139,0.053,2.984,0.034,335.128
6,pytorch,fp32,4,78.337,0.332,93.655,0.103,3.082,0.019,324.479
7,pytorch,fp32,8,77.988,0.132,93.581,0.232,3.043,0.065,328.733


In [ ]:
bench_path = PROJECT_ROOT / "resultsv2" / "latency_bench" / "pytorch_fp32_in8b_cuda_bs1.json"
with open(bench_path) as f:
    bench = json.load(f)
bench_std = np.std(bench["latencies_ms"], ddof=1)

avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {bench_std:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "ptq_torch_avg_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")